### 멀티모달 RAG
- 01.multimodal 에서 추출한 텍스트와 이미지요약본을 VectorDB에 임베딩하고 

    사용자의 질문에 맞춰 관련 컨텍스트를 검색(Retrival)한뒤 LLM을 이용해서 최종 답변을 생성(Generation)

In [1]:
import os
import fitz
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# 환경변수 로드 및 openai 클라이언트 생성
load_dotenv(override=True)
client = OpenAI()

In [3]:
# 벡터 Db 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db')
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('OPENAI_API_KEY'),
    model_name = 'text-embedding-3-small'
)
collection = chroma_client.get_or_create_collection(name='multimodal_rag',embedding_function=openai_ef)
print(f'설정완료 현재 컬렉션의 크기 : {collection.count()}')

설정완료 현재 컬렉션의 크기 : 0


### 데이터 임베딩(텍스트 + 이미지 요약본)
- pdf의 텍스트와 이미지 요약본을 모두 VectorDB에 삽입(동일한 벡터공간에 텍스트와 이미지 요약본이 함께 존재)

In [ ]:
import base64
if collection.count() == 0:
    pdf_path = 'sample_paper.pdf'
    doc = fitz.open(pdf_path)
    documents = []
    metatdatas = []    
    ids = []
    
    for page_num in range(min(5,len(doc))):  # 테스트용으로 5페이지만 처리        
        page = doc.load_page(page_num)
        # 텍스트 추출
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metatdatas.append({'page':page_num+1,'type':'text'})
            ids.append(f'text_page_{page_num+1}')
        # 이미지 요약 후 추출
        image_list = page.get_images(full=True)
        if image_list:
            xref = image_list[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            if ext.lower() == 'jb2': ext = 'jpeg'
            mime_type = f'image/{ext}' if ext.lower() != "jpg" else 'image/jpeg'
            base64_image = base64.b64encode(image_bytes).decode('utf-8')
            try:
                response = client.chat.completions.create(
                    model = 'gpt-5.4-nano',
                    messages=[
                        {'role':'user', 
                        'content': [
                            {'type':'text', 'text':'이 이미지에 대한 상세한 설명을 작성해 주세요. 전문적이고 구체적으로 분석해야 합니다.'},
                            {'type':'image_url', 'image_url':{
                                'url':f'data:{mime_type};base64,{base64_image}'
                                }
                            }
                        ]
                        }
                    ],
                    max_completion_tokens=300
                )
                summary = response.choices[0].message.content
                documents.append(summary)
                metatdatas.append({
                    'type':'image_summary','page':page_num+1
                })
                ids.append(f'image_summary_page_{page_num+1}')
            except Exception as e:
                pass
    if documents:
        collection.add(documents=documents,metadatas=metatdatas,ids=ids)
        print('VectorDB에 데이터 임베딩 및 저장 완료')
else:
    print(f'vectorDB에 이미 {collection.count()}개 데이터가 존재합니다')
